In [ ]:
import pandas as pd
from google.colab import userdata

path = userdata.get('path_to_dataset')


# Предположим, у вас есть DataFrame с товарами
df_test = pd.read_csv(path)
df_test = df_test.rename(columns={'NameFull':'description'})
df_test.head()

,index,description
0,45983,"Колесо для тачки FIT 77556 16""x4"""
1,16660,Комплект колодок тормозные дисковые передние A...
2,10302,Труба бесшовная горячедеформированная В; с про...
3,8247,Теплообменник кожухотрубчатый горизонтальный Д...
4,3068,Устройство дистанционной защиты Schneider Elec...


In [ ]:
COL_NAME = 'description'

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 91.4 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/LaBSE')
model.encode('Hellow')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

array([-3.01677734e-02, -6.22088499e-02, -1.91544537e-02, -5.84574006e-02,
        3.48032676e-02, -2.19501946e-02, -2.67367959e-02,  1.48179447e-02,
       -5.39121144e-02, -2.04572864e-02,  4.76542190e-02, -4.18084264e-02,
        2.74103694e-02, -4.87614796e-02, -5.70070930e-02, -5.29825836e-02,
       -1.95945613e-02,  1.52311856e-02,  5.05207628e-02,  2.31476296e-02,
        1.55408867e-03,  3.76164392e-02, -2.27637775e-03, -4.33193706e-02,
       -2.97319144e-02, -1.97394043e-02,  9.41952597e-03,  1.70101393e-02,
        8.96180701e-03, -4.77510691e-02, -5.70631064e-02,  5.68229929e-02,
       -4.00346108e-02,  4.76326086e-02, -3.91101055e-02, -5.24742343e-02,
       -5.17100468e-02,  4.72228155e-02,  2.63481773e-02, -6.21588528e-02,
        3.46467905e-02,  3.93267274e-02,  1.10207044e-03, -2.97229048e-02,
        4.39064577e-02,  5.47120869e-02,  4.87774573e-02, -4.96652797e-02,
       -5.25980070e-02, -3.37322578e-02, -4.44487818e-02, -5.74218296e-02,
       -2.03447957e-02, -

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
from faiss import read_index
import pandas as pd



# Read the dataset and prepare the subject pack
path = '/content/drive/MyDrive/AIWeekKrasnoyarsk/CaseLLMSearch/data_fixed.csv'
df = pd.read_csv(path)
df = df.rename(columns={'NameFull':'description'})

subj_pack = df[COL_NAME].tolist()
id_pack = df.index.tolist()

# Pydantic model for the response
class SearchResult(BaseModel):
    model_name: str
    k_nearest: List[str]
    k_nearest_ids: List[int]

class SearchResponse(BaseModel):
    results: List[SearchResult]

# Load the tokenizer and model for laBSE
model_name_bse = 'sentence-transformers/LaBSE'
model = SentenceTransformer(model_name_bse)
index_ = read_index("/content/drive/MyDrive/AIWeekKrasnoyarsk/CaseLLMSearch/LaBSE_FAISS.index")

def search(text: str = '', k: int = 5):
    # e5
    encoded_input = text

    with torch.no_grad():
        model_output = model.encode(encoded_input)
        query_vector = model_output
    # Search in e5 index
    query_vector = np.array([query_vector])

    D, I = index_.search(query_vector, k)
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    result = SearchResult(model_name=model_name_bse, k_nearest=results, k_nearest_ids=ids)


    # Return combined results from both models
    return SearchResponse(results=[result])


In [ ]:
search('анализатор')

SearchResponse(results=[SearchResult(model_name='sentence-transformers/LaBSE', k_nearest=['Анализатор вольтамперометрический TA-Lab', 'Анализатор титрометрический лабораторный для определения влагосодержания Цвет АТЛ-11-01', 'Опрыскиватель', 'Газоанализатор переносной Газ-Аналитик ОКА-92-О2 с выносным блоком датчика(кабель 6м)', 'Анализатор вибрации Электронприбор Топаз-В портативный анализатор вибрации 255х175х75мм'], k_nearest_ids=[27125, 27126, 4997, 3800, 37106])])

In [ ]:
import numpy as np
from typing import List, Dict, Tuple
import pandas as pd

def calculate_benchmark_metrics(test_df: pd.DataFrame,
                               search_function,
                               k: int = 5) -> Dict[str, float]:
    """
    Рассчитывает метрики качества поиска синонимов

    Args:
        test_df: DataFrame с колонками 'index' и 'description'
        search_function: функция поиска, принимающая текст и возвращающая SearchResponse
        k: количество возвращаемых результатов

    Returns:
        Словарь с метриками для каждой модели
    """

    metrics = {
        'multilingual-e5-base': {
            'precision@1': 0,
            'precision@3': 0,
            'precision@5': 0,
            'mrr': 0,
            'found_in_top_k': 0
        },
        model_name_bse: {
            'precision@1': 0,
            'precision@3': 0,
            'precision@5': 0,
            'mrr': 0,
            'found_in_top_k': 0
        }
    }

    total_samples = len(test_df)

    for idx, row in test_df.iterrows():
        original_index = row['index']
        query_text = row['description']

        # Получаем результаты поиска
        search_response = search_function(text=query_text, k=k)

        for result in search_response.results:
            model_name = result.model_name
            found_indices = result.k_nearest_ids

            # Precision@1, @3, @5
            for n in [1, 3, 5]:
                if n <= k:
                    top_n = found_indices[:n]
                    if original_index in top_n:
                        metrics[model_name][f'precision@{n}'] += 1

            # MRR (Mean Reciprocal Rank)
            for rank, found_idx in enumerate(found_indices, 1):
                if found_idx == original_index:
                    metrics[model_name]['mrr'] += 1.0 / rank
                    metrics[model_name]['found_in_top_k'] += 1
                    break

    # Нормализуем метрики
    for model_name in metrics:
        for metric in ['precision@1', 'precision@3', 'precision@5', 'found_in_top_k']:
            if metric in metrics[model_name]:
                metrics[model_name][metric] /= total_samples

        # Нормализуем MRR
        metrics[model_name]['mrr'] /= total_samples

    return metrics

def print_metrics(metrics: Dict[str, Dict]):
    """Красиво выводит метрики"""
    print("=" * 60)
    print("БЕНЧМАРК ТОЧНОСТИ ПОИСКА СИНОНИМОВ")
    print("=" * 60)

    for model_name, model_metrics in metrics.items():
        print(f"\nМодель: {model_name}")
        print("-" * 40)
        print(f"Precision@1:  {model_metrics['precision@1']:.4f}")
        print(f"Precision@3:  {model_metrics['precision@3']:.4f}")
        print(f"Precision@5:  {model_metrics['precision@5']:.4f}")
        print(f"MRR:          {model_metrics['mrr']:.4f}")
        print(f"Found in top-{len(model_metrics)-4}: {model_metrics['found_in_top_k']:.4f}")

# Пример использования
def run_benchmark():
    # Загружаем тестовые данные
    test_df = pd.read_csv("test_data.csv")  # Ваш test_df

    # Запускаем бенчмарк
    metrics = calculate_benchmark_metrics(test_df, search, k=5)

    # Выводим результаты
    print_metrics(metrics)

    return metrics

# Дополнительные полезные функции

def analyze_failures(test_df: pd.DataFrame, search_function, k: int = 5) -> pd.DataFrame:
    """
    Анализирует случаи, когда синоним не найден
    """
    failures = []

    for idx, row in test_df.iterrows():
        original_index = row['index']
        query_text = row['description']

        search_response = search_function(text=query_text, k=k)

        for result in search_response.results:
            if original_index not in result.k_nearest_ids:
                failures.append({
                    'model': result.model_name,
                    'query_index': original_index,
                    'query_text': query_text[:100],  # первые 100 символов
                    'found_indices': result.k_nearest_ids,
                    'found_texts': [text[:50] for text in result.k_nearest]  # первые 50 символов
                })

    return pd.DataFrame(failures)

def calculate_confusion_matrix(test_df: pd.DataFrame, search_function, k: int = 5):
    """
    Создает confusion matrix для анализа ошибок
    """
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    all_true = []
    all_pred = []

    for idx, row in test_df.iterrows():
        original_index = row['index']
        query_text = row['description']

        search_response = search_function(text=query_text, k=1)  # только top-1

        for result in search_response.results:
            all_true.append(original_index)
            all_pred.append(result.k_nearest_ids[0])

    # Создаем confusion matrix
    cm = confusion_matrix(all_true, all_pred)

    # Визуализация
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix для поиска синонимов')
    plt.ylabel('Истинные индексы')
    plt.xlabel('Предсказанные индексы')
    plt.show()

    return cm

# Если нужно добавить веса для разных типов товаров
def weighted_benchmark(test_df: pd.DataFrame, search_function,
                      weights_column: str = None, k: int = 5):
    """
    Взвешенный бенчмарк с учетом важности разных товаров
    """
    if weights_column and weights_column in test_df.columns:
        weights = test_df[weights_column].values
    else:
        weights = np.ones(len(test_df))

    metrics = {
        'multilingual-e5-base': {
            'precision@1': 0,
            'precision@3': 0,
            'precision@5': 0,
            'mrr': 0,
            'found_in_top_k': 0
        },
        'sbert_large_mt_nlu_ru': {
            'precision@1': 0,
            'precision@3': 0,
            'precision@5': 0,
            'mrr': 0,
            'found_in_top_k': 0
        }
    }

    total_weight = weights.sum()

    for idx, (_, row) in enumerate(test_df.iterrows()):
        original_index = row['index']
        query_text = row['description']
        weight = weights[idx]

        search_response = search_function(text=query_text, k=k)

        for result in search_response.results:
            model_name = result.model_name
            found_indices = result.k_nearest_ids

            for n in [1, 3, 5]:
                if n <= k:
                    top_n = found_indices[:n]
                    if original_index in top_n:
                        metrics[model_name][f'precision@{n}'] += weight

            for rank, found_idx in enumerate(found_indices, 1):
                if found_idx == original_index:
                    metrics[model_name]['mrr'] += (1.0 / rank) * weight
                    metrics[model_name]['found_in_top_k'] += weight
                    break

    # Нормализация по весам
    for model_name in metrics:
        for metric in ['precision@1', 'precision@3', 'precision@5', 'found_in_top_k']:
            if metric in metrics[model_name]:
                metrics[model_name][metric] /= total_weight

        metrics[model_name]['mrr'] /= total_weight

    return metrics

# Основной запуск
# Запуск простого бенчмарка
metrics = calculate_benchmark_metrics(df_test, search, k=5)
print_metrics(metrics)

# Анализ ошибок
failures_df = analyze_failures(df_test, search, k=5)
print(f"\nВсего ошибок: {len(failures_df)}")

if len(failures_df) > 0:
    print("\nПримеры ошибок:")
    print(failures_df.head())

БЕНЧМАРК ТОЧНОСТИ ПОИСКА СИНОНИМОВ

Модель: multilingual-e5-base
----------------------------------------
Precision@1:  0.0000
Precision@3:  0.0000
Precision@5:  0.0000
MRR:          0.0000
Found in top-1: 0.0000

Модель: sentence-transformers/LaBSE
----------------------------------------
Precision@1:  0.9480
Precision@3:  0.9740
Precision@5:  0.9800
MRR:          0.9612
Found in top-1: 0.9800

Всего ошибок: 10

Примеры ошибок:
                         model  query_index  \
0  sentence-transformers/LaBSE        14151   
1  sentence-transformers/LaBSE        19263   
2  sentence-transformers/LaBSE         2624   
3  sentence-transformers/LaBSE         3063   
4  sentence-transformers/LaBSE        32661   

                                          query_text  \
0                          Шкурка шлифовальная ткань   
1                           Блок сцепления в главный   
2                      перепада Теплоприбор 0.5% Exd   
3  Терминал НПП RJ45(Eth)х2;COM1;COM2;COM3;RS-485...   
4   

In [ ]:
# Анализ ошибок
failures_df = analyze_failures(df_test, search, k=5)
print(f"\nВсего ошибок: {len(failures_df)}")

if len(failures_df) > 0:
    print("\nПримеры ошибок:")
    print(failures_df.head())


Всего ошибок: 220

Примеры ошибок:
                   model  query_index  \
0   multilingual-e5-base        10302   
1   multilingual-e5-base         9103   
2  sbert_large_mt_nlu_ru         9103   
3   multilingual-e5-base        48757   
4   multilingual-e5-base        12206   

                                          query_text  \
0  Труба бесшовная горячедеформированная В; с про...   
1  Свая железобетонная С70.30-9 B10 W4 F100 7000м...   
2  Свая железобетонная С70.30-9 B10 W4 F100 7000м...   
3  Модуль памяти HP 1Rx8 13L77AA DDR4 SO-DIMM 8Gb...   
4  Компенсатор сальниковый односторонний DN1000 P...   

                         found_indices  \
0  [11106, 11078, 10817, 10675, 10761]   
1      [9116, 9113, 16278, 9100, 9101]   
2       [9104, 9079, 9085, 9105, 9118]   
3       [2989, 2996, 2997, 2995, 2965]   
4  [12214, 12213, 12210, 12208, 12209]   

                                         found_texts  
0  [Труба бесшовная горячедеформированная В 32х2....  
1  [Свая железобе

In [ ]:
failures_df.head()

,model,query_index,query_text,found_indices,found_texts
0,multilingual-e5-base,10302,Труба бесшовная горячедеформированная В; с про...,"[11106, 11078, 10817, 10675, 10761]",[Труба бесшовная горячедеформированная В 32х2....
1,multilingual-e5-base,9103,Свая железобетонная С70.30-9 B10 W4 F100 7000м...,"[9116, 9113, 16278, 9100, 9101]",[Свая железобетонная С70.35-10 B10 W4 F100 700...
2,sbert_large_mt_nlu_ru,9103,Свая железобетонная С70.30-9 B10 W4 F100 7000м...,"[9104, 9079, 9085, 9105, 9118]",[Свая железобетонная С70.30-8 B10 W4 F100 7000...
3,multilingual-e5-base,48757,Модуль памяти HP 1Rx8 13L77AA DDR4 SO-DIMM 8Gb...,"[2989, 2996, 2997, 2995, 2965]",[Модуль процессора Siemens 6AG1315-2AH14-2AY0 ...
4,multilingual-e5-base,12206,Компенсатор сальниковый односторонний DN1000 P...,"[12214, 12213, 12210, 12208, 12209]",[Компенсатор сальниковый односторонний DN600 P...


In [ ]:
failures_df[failures_df['model']=='sbert_large_mt_nlu_ru'].iloc[0]['query_text']

'Свая железобетонная С70.30-9 B10 W4 F100 7000мм 300х300мм Серия 1.011.1-10 АО Завод ЖБИ-3'

In [ ]:
failures_df[failures_df['model']=='sbert_large_mt_nlu_ru'].iloc[0]['found_texts']

['Свая железобетонная С70.30-8 B10 W4 F100 7000мм 30',
 'Свая железобетонная С40.30-2 B10 W4 F100 4000мм 30',
 'Свая железобетонная С50.30-2 B10 W4 F100 5000мм 30',
 'Свая железобетонная С70.30-6 B10 W4 F100 7000мм 30',
 'Свая железобетонная С90.30-5 B10 W4 F100 9000мм 30']